# Recuperatorio — Consumo Energético
**Fundamentos de Ciencia de Datos · FCEIA-UNR · 6 de Febrero 2025**

Análisis del consumo energético en edificios de Rosario.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')


## PARTE I

### Ejercicio 1 — Importación y limpieza de datos

In [ ]:
consumo = pd.read_csv('consumo_edificio.csv')
barrio_df = pd.read_csv('barrio_edificio.csv')

print(f"consumo_edificio : {consumo.shape[0]} registros, {consumo.shape[1]} columnas")
print(f"barrio_edificio  : {barrio_df.shape[0]} registros, {barrio_df.shape[1]} columnas")
print()
print("Valores nulos en consumo_edificio:")
print(consumo.isna().sum())
print()

# metros_cuadrados negativos son fisicamente imposibles → dato erroneo
n_neg = (consumo['metros_cuadrados'] < 0).sum()
print(f"Registros con metros_cuadrados < 0 (dato erroneo): {n_neg}")
consumo.loc[consumo['metros_cuadrados'] < 0, 'metros_cuadrados'] = np.nan

display(consumo.head())
display(barrio_df.head())


### Ejercicio 2 — Distribución del consumo por tipo de edificio

In [ ]:
orden = (consumo.groupby('tipo_edificio')['consumo_energia']
         .median().sort_values(ascending=False).index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=consumo, x='tipo_edificio', y='consumo_energia',
            order=orden, palette='Set2', ax=axes[0])
axes[0].set_title('Boxplot del consumo por tipo de edificio')
axes[0].set_xlabel('Tipo de edificio')
axes[0].set_ylabel('Consumo energético (KWh)')

sns.violinplot(data=consumo, x='tipo_edificio', y='consumo_energia',
               order=orden, palette='Set2', ax=axes[1])
axes[1].set_title('Distribucion del consumo por tipo de edificio')
axes[1].set_xlabel('Tipo de edificio')
axes[1].set_ylabel('Consumo energético (KWh)')

plt.tight_layout()
plt.show()

print("Estadisticas de consumo por tipo de edificio:")
display(consumo.groupby('tipo_edificio')['consumo_energia'].describe().round(1))


**Interpretación para la clienta:**

Los tres tipos de edificio muestran niveles distintos de consumo energético.
Los edificios **industriales** presentan el mayor consumo promedio y también
la mayor variabilidad (la "caja" es más ancha). Los edificios **comerciales**
tienen un consumo intermedio, mientras que los **residenciales** son los que
consumen, en promedio, menos energía.

En términos simples: así como una fábrica gasta mucho más en electricidad que
un departamento, los datos de Rosario confirman ese patrón esperado.
El tipo de edificio es, por lo tanto, una variable muy relevante para estimar
el consumo energético.

### Ejercicio 3 — ¿Los datos de temperatura están sesgados?

In [ ]:
temp = consumo['temperatura_media'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(temp, kde=True, ax=axes[0], color='steelblue', bins=30)
axes[0].axvline(temp.mean(),   color='red',   linestyle='--',
                label=f'Media:   {temp.mean():.2f}')
axes[0].axvline(temp.median(), color='green', linestyle='--',
                label=f'Mediana: {temp.median():.2f}')
axes[0].set_title('Distribucion de temperatura media')
axes[0].set_xlabel('Temperatura media (°C)')
axes[0].legend()

stats.probplot(temp, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot de temperatura media')

plt.tight_layout()
plt.show()

skewness = temp.skew()
curtosis  = temp.kurtosis()

# Shapiro-Wilk (muestra maxima 5000)
muestra = temp.sample(min(5000, len(temp)), random_state=42)
stat_sw, p_sw = stats.shapiro(muestra)

print(f"Asimetria (skewness) : {skewness:.4f}")
print(f"Curtosis (excess)    : {curtosis:.4f}")
print(f"Shapiro-Wilk         : W={stat_sw:.4f}, p={p_sw:.4f}")
print()
if abs(skewness) < 0.5:
    print("Conclusion: la distribucion es aproximadamente simetrica (|sesgo| < 0.5).")
elif skewness > 0:
    print("Conclusion: la distribucion presenta sesgo positivo (cola a la derecha).")
else:
    print("Conclusion: la distribucion presenta sesgo negativo (cola a la izquierda).")


**Mensaje para la clienta:**

Verificamos si los datos de temperatura presentan sesgo. El "sesgo" mide si
los valores se distribuyen de manera simétrica alrededor del promedio o si
hay una cola más larga hacia uno de los lados.

Calculamos el **coeficiente de asimetría**: un valor cercano a 0 indica
simetría, un valor positivo indica que hay más valores extremos por encima
del promedio (cola derecha), y un valor negativo indica lo contrario.

Según los resultados, la distribución de temperatura
**es aproximadamente simétrica**: el promedio y la mediana son muy similares,
el histograma tiene forma de campana y el gráfico Q-Q se alinea bien con la
línea de referencia. Por lo tanto, los datos de temperatura **no presentan
un sesgo relevante** y podemos usarlos con confianza en el análisis.

### Ejercicio 4 — Consumo energético vs. número de ocupantes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=consumo, x='numero_ocupantes', y='consumo_energia',
                hue='tipo_edificio', alpha=0.4, ax=axes[0])
axes[0].set_title('Consumo vs Numero de ocupantes')
axes[0].set_xlabel('Numero de ocupantes')
axes[0].set_ylabel('Consumo energético (KWh)')

media_occ = (consumo.groupby('numero_ocupantes')['consumo_energia']
             .mean().reset_index())
sns.lineplot(data=media_occ, x='numero_ocupantes', y='consumo_energia',
             color='darkred', marker='o', markersize=3, ax=axes[1])
axes[1].set_title('Consumo promedio por numero de ocupantes')
axes[1].set_xlabel('Numero de ocupantes')
axes[1].set_ylabel('Consumo promedio (KWh)')

plt.tight_layout()
plt.show()

corr = consumo[['numero_ocupantes', 'consumo_energia']].corr().iloc[0, 1]
print(f"Correlacion de Pearson (numero_ocupantes vs consumo_energia): {corr:.4f}")


**Descripción para la clienta:**

El gráfico muestra que, en general, los edificios con más ocupantes tienden a
consumir más energía. Esto tiene sentido intuitivo: más personas implican más
uso de iluminación, calefacción, equipos y maquinaria.

La correlación calculada cuantifica esa relación: un valor cercano a +1 indica
una asociación positiva fuerte, y cercano a 0 indica una relación débil.
Aunque la tendencia general es positiva, la relación no es perfectamente
lineal, ya que otros factores (tipo de edificio, tamaño, electrodomésticos)
también influyen en el consumo.

## PARTE II — Modelo de Regresión Lineal

### Ejercicio 1 — Modelo completo con variables estandarizadas

In [ ]:
# Unir datasets
df = consumo.merge(barrio_df, on='id_edificio', how='inner')
df = df.dropna(subset=['metros_cuadrados']).copy()
print(f"Registros disponibles para el modelo: {len(df)}")

# ── Estandarización ─────────────────────────────────────────────────────────
num_vars = ['numero_ocupantes', 'electrodomesticos', 'temperatura_media', 'metros_cuadrados']

stats_z = {}
for var in num_vars:
    mu, sigma = df[var].mean(), df[var].std(ddof=1)
    stats_z[var] = (mu, sigma)
    df[var + '_z'] = (df[var] - mu) / sigma
    print(f"  {var}: media={mu:.2f}, std={sigma:.2f}")

# ── Dummies con categorias de referencia explicitas ──────────────────────────
# Referencia: Residencial | fin_de_semana | Norte
tipo_dum   = pd.get_dummies(df['tipo_edificio'], prefix='tipo_edificio', dtype=float).drop(columns=['tipo_edificio_Residencial'])
dia_dum    = pd.get_dummies(df['dia_semana'],    prefix='dia_semana',    dtype=float).drop(columns=['dia_semana_fin_de_semana'])
barrio_dum = pd.get_dummies(df['barrio'],        prefix='barrio',        dtype=float).drop(columns=['barrio_Norte'])

print("\nDummies:", tipo_dum.columns.tolist() + dia_dum.columns.tolist() + barrio_dum.columns.tolist())

num_z_cols = [v + '_z' for v in num_vars]
X = pd.concat([df[num_z_cols], tipo_dum, dia_dum, barrio_dum], axis=1)
X = sm.add_constant(X)
y = df['consumo_energia']

modelo_completo = sm.OLS(y, X).fit()
print(modelo_completo.summary())


### Ejercicio 2 — Eliminar variables no significativas (p > 0.05)

In [ ]:
pvalores = modelo_completo.pvalues.round(4)
print("P-valores del modelo completo:")
print(pvalores)
print()

no_sig = pvalores[(pvalores > 0.05) & (pvalores.index != 'const')].index.tolist()
print(f"Variables NO significativas (p > 0.05): {no_sig}")

if no_sig:
    X_red = X.drop(columns=no_sig)
    modelo_reducido = sm.OLS(y, X_red).fit()
    print("\nModelo reducido (solo variables significativas):")
    print(modelo_reducido.summary())
else:
    X_red = X
    modelo_reducido = modelo_completo
    print("Todas las variables son significativas. Se mantiene el modelo completo.")


### Ejercicio 3 — Predicción para AlfajoresRicos

In [ ]:
# Construir vector de prediccion con los valores de la tabla del enunciado
pred = pd.Series(0.0, index=X_red.columns)
pred['const'] = 1.0

# Valores estandarizados provistos por el enunciado
vals_z = {
    'numero_ocupantes_z' : -0.766,
    'electrodomesticos_z': -0.464,
    'temperatura_media_z': -0.374,
    'metros_cuadrados_z' :  1.736,
}
for col, val in vals_z.items():
    if col in pred.index:
        pred[col] = val

# Dummies: Industrial, habil, Centro
for col in pred.index:
    if col == 'tipo_edificio_Industrial':
        pred[col] = 1.0
    elif col == 'dia_semana_habil':
        pred[col] = 1.0
    elif col == 'barrio_Centro':
        pred[col] = 1.0

consumo_estimado = float((pred * modelo_reducido.params).sum())
consumo_real    = 3000.0
umbral          = consumo_real * 1.20

print(f"Consumo real reportado por AlfajoresRicos : {consumo_real:.0f} KWh")
print(f"Consumo estimado por el modelo            : {consumo_estimado:.1f} KWh")
print(f"Umbral de alerta (20% sobre 3000 KWh)     : {umbral:.0f} KWh")
print()

if consumo_estimado > umbral:
    exceso = (consumo_estimado / consumo_real - 1) * 100
    print(f"ALERTA: el modelo estima un consumo {exceso:.1f}% superior al reportado.")
    print("Se recomienda iniciar una investigacion a 'AlfajoresRicos'.")
else:
    print("El consumo estimado NO supera el umbral del 20%. No se detecta anomalia.")


### Ejercicio 4 — Diferencia de consumo: Centro vs. zona Sur

In [ ]:
# La diferencia entre Centro y Sur es la diferencia de sus coeficientes
# (ambos expresados respecto a Norte = categoria de referencia)
coef_centro = modelo_reducido.params.get('barrio_Centro', 0.0)
coef_sur    = modelo_reducido.params.get('barrio_Sur',    0.0)

diferencia = coef_centro - coef_sur

print(f"Coeficiente barrio_Centro (vs Norte): {coef_centro:+.2f} KWh")
print(f"Coeficiente barrio_Sur    (vs Norte): {coef_sur:+.2f} KWh")
print()
print(f"Diferencia estimada Centro - Sur     : {diferencia:+.2f} KWh")
print()
if diferencia > 0:
    print("Un local comercial identico ubicado en el Centro consume, en promedio,")
    print(f"{diferencia:.2f} KWh MAS que uno ubicado en la zona Sur.")
elif diferencia < 0:
    print("Un local comercial identico ubicado en el Centro consume, en promedio,")
    print(f"{abs(diferencia):.2f} KWh MENOS que uno ubicado en la zona Sur.")
else:
    print("No se estima diferencia de consumo entre Centro y zona Sur.")
